# Week 4: ML 工作流与线性模型

## 学习目标

1. 理解机器学习工作流的完整流程
2. 掌握线性回归和逻辑回归的基本原理
3. 学会使用 Scikit-learn 进行模型训练和预测
4. 理解模型评估指标的含义

## 1. ML 工作流概述

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix

np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("ML 工具已加载")

### 1.1 工作流步骤

1. **数据准备**：读取、清洗、特征工程
2. **数据分割**：训练集 / 验证集 / 测试集
3. **模型选择**：选择合适的算法
4. **模型训练**：拟合数据
5. **模型评估**：计算性能指标
6. **模型优化**：调参、改进
7. **模型部署**：应用到生产环境

In [ ]:
# 生成模拟数据
n_samples = 200

# 特征：温度、是否周末、是否假期
temperature = np.random.uniform(5, 35, n_samples)
is_weekend = np.random.randint(0, 2, n_samples)
is_holiday = np.random.randint(0, 2, n_samples)

# 目标：需求量（带噪声）
# 需求 = 10 + 1.5*温度 - 5*周末 + 10*假期 + 噪声
true_demand = 10 + 1.5 * temperature - 5 * is_weekend + 10 * is_holiday
noise = np.random.normal(0, 3, n_samples)
demand = true_demand + noise

# 创建 DataFrame
df = pd.DataFrame({
    'temperature': temperature,
    'is_weekend': is_weekend,
    'is_holiday': is_holiday,
    'demand': demand
})

print("模拟数据已生成")
print(df.head())
print(f"\n数据形状: {df.shape}")

## 2. 线性回归

### 2.1 数据准备

In [ ]:
# 特征和目标
X = df[['temperature', 'is_weekend', 'is_holiday']].values
y = df['demand'].values

# 训练集 / 测试集分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")

### 2.2 模型训练

In [ ]:
# 训练线性回归模型
model = LinearRegression()
model.fit(X_train, y_train)

# 模型参数
print("线性回归模型参数")
print("=" * 40)
print(f"截距 (intercept): {model.intercept_:.2f}")
print(f"系数 (coefficients):")
for i, coef in enumerate(model.coef_):
    feature_name = ['temperature', 'is_weekend', 'is_holiday'][i]
    print(f"  {feature_name}: {coef:.2f}")

print("\n真实参数：")
print("  intercept: 10")
print("  temperature: 1.5")
print("  is_weekend: -5")
print("  is_holiday: 10")

### 2.3 模型评估

In [ ]:
# 预测
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# 评估指标
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("模型评估结果")
print("=" * 40)
print(f"训练集 MSE: {train_mse:.2f}")
print(f"测试集 MSE: {test_mse:.2f}")
print(f"训练集 R²: {train_r2:.4f}")
print(f"测试集 R²: {test_r2:.4f}")

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 训练集
axes[0].scatter(y_train, y_pred_train, alpha=0.5)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('真实值')
axes[0].set_ylabel('预测值')
axes[0].set_title(f'训练集 (R² = {train_r2:.3f})')
axes[0].grid(True, alpha=0.3)

# 测试集
axes[1].scatter(y_test, y_pred_test, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('真实值')
axes[1].set_ylabel('预测值')
axes[1].set_title(f'测试集 (R² = {test_r2:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 逻辑回归

### 3.1 分类问题

将需求分为"高需求"和"低需求"两类

In [ ]:
# 创建分类目标
demand_threshold = np.median(df['demand'])
df['high_demand'] = (df['demand'] > demand_threshold).astype(int)

print(f"需求阈值: {demand_threshold:.2f}")
print(f"高需求比例: {df['high_demand'].mean():.2%}")

In [ ]:
# 准备数据
X_cls = df[['temperature', 'is_weekend', 'is_holiday']].values
y_cls = df['high_demand'].values

# 分割数据
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

# 训练逻辑回归
clf = LogisticRegression()
clf.fit(X_train_cls, y_train_cls)

# 预测
y_pred_cls = clf.predict(X_test_cls)
y_prob_cls = clf.predict_proba(X_test_cls)[:, 1]

print("逻辑回归模型参数")
print("=" * 40)
print(f"截距: {clf.intercept_[0]:.2f}")
for i, coef in enumerate(clf.coef_[0]):
    feature_name = ['temperature', 'is_weekend', 'is_holiday'][i]
    print(f"  {feature_name}: {coef:.2f}")

In [ ]:
# 评估
accuracy = accuracy_score(y_test_cls, y_pred_cls)
conf_mat = confusion_matrix(y_test_cls, y_pred_cls)

print("分类评估结果")
print("=" * 40)
print(f"准确率: {accuracy:.2%}")
print(f"\n混淆矩阵:")
print(conf_mat)

In [ ]:
# 可视化决策边界（固定其他变量）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 温度 vs 预测概率（工作日，非假期）
temp_range = np.linspace(5, 35, 100)
X_vis = np.zeros((100, 3))
X_vis[:, 0] = temp_range
X_vis[:, 1] = 0  # 工作日
X_vis[:, 2] = 0  # 非假期

probs = clf.predict_proba(X_vis)[:, 1]

axes[0].plot(temp_range, probs, linewidth=2)
axes[0].axhline(0.5, color='red', linestyle='--', label='决策阈值')
axes[0].set_xlabel('温度 (°C)')
axes[0].set_ylabel('高需求概率')
axes[0].set_title('逻辑回归：温度 vs 高需求概率\n(工作日，非假期)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 混淆矩阵可视化
import seaborn as sns
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_xlabel('预测')
axes[1].set_ylabel('真实')
axes[1].set_title('混淆矩阵')

plt.tight_layout()
plt.show()

## 4. 特征工程

In [ ]:
# 特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("标准化前后的特征统计")
print("=" * 40)
print("原始数据:")
print(f"  均值: {X_train.mean(axis=0)}")
print(f"  标准差: {X_train.std(axis=0)}")
print("\n标准化后:")
print(f"  均值: {X_train_scaled.mean(axis=0)}")
print(f"  标准差: {X_train_scaled.std(axis=0)}")

In [ ]:
# 添加交互特征
df['temp_weekend'] = df['temperature'] * df['is_weekend']
df['temp_holiday'] = df['temperature'] * df['is_holiday']

print("添加交互特征后的数据:")
print(df.head())

## 5. Research Thinking

### 问题 1：相关 vs 因果

线性回归系数能告诉我们因果关系吗？

**回答：**

不能！回归系数只表示关联性。

**例子：**
- 温度系数为正，不意味着提高温度就能增加需求
- 可能存在混杂变量（如季节、节假日）
- 建立因果需要控制所有混杂变量或使用因果推断方法

### 问题 2：过拟合 vs 欠拟合

如何判断模型是否过拟合或欠拟合？

**回答：**

- **欠拟合**：训练集和测试集误差都很高
- **过拟合**：训练集误差低，测试集误差高
- **合适**：训练集和测试集误差都低且接近

### 问题 3：特征选择

如何选择有意义的特征？

**回答：**

1. **领域知识**：理解业务逻辑
2. **相关性分析**：特征与目标的关联
3. **特征重要性**：模型的系数或重要性得分
4. **正则化**：L1/L2 正则化自动选择
5. **交叉验证**：比较不同特征集的性能

## 6. 练习

### 练习 1
添加更多特征（如温度的平方项），观察模型性能变化。

In [ ]:
# 你的代码


### 练习 2
比较标准化前后的模型性能。

In [ ]:
# 你的代码


### 练习 3
使用真实数据集训练模型，并进行完整的工作流演示。

In [ ]:
# 你的代码


## 7. 总结

### 本周学习要点

1. **ML 工作流**：数据准备 → 训练 → 评估 → 优化
2. **线性回归**：预测连续值，最小化 MSE
3. **逻辑回归**：分类问题，预测概率
4. **特征工程**：标准化、交互特征

### 关键洞察

- 回归系数表示关联，不是因果
- 训练集和测试集分离防止过拟合
- 特征工程对模型性能至关重要